In [ ]:
1. What is a Subgraph?
A Subgraph is essentially a graph that is embedded and executed as a single node within another "parent" graph 
This allows you to treat a complex sequence of steps as a modular unit.

2. Why Use Subgraphs?
Modularity: Breaks down complex AI workflows into smaller, manageable functions 
Reusability: A subgraph (e.g., a "coding agent") can be used by multiple different parent nodes 
Maintainability: Easier to debug specific parts of a large system without affecting the whole 
Failure Isolation: If a subgraph fails, it doesn’t necessarily crash the entire parent graph 
State Separation: Subgraphs can maintain their own internal state, preventing "state bloat" in the parent graph 


3. Implementation Mechanisms
There are two primary ways to implement subgraphs in LangGraph:
Invoking from a Node: The parent graph calls the subgraph manually inside a node function. This keeps states completely isolated 
Adding as a Node: The subgraph is added directly to the parent graph using .add_node("name", subgraph). This allows for shared state keys 

***************************************************************************************************************************************************
❓ Interview Questions
What is the primary architectural benefit of using subgraphs in a Multi-Agent System?

Answer: They provide encapsulation. Each agent can have its own internal logic, tools, memory, and state, which prevents the parent graph 
        from becoming overly complex and hard to manage 

How does LangGraph handle state between a parent graph and a subgraph?

Answer: It depends on the implementation. You can either have Isolated States (where the subgraph has its own schema)
        or Shared States (where the subgraph operates on a subset of the parent's state keys) 


Can you trace/monitor a subgraph independently?

Answer: Yes. Using tools like LangSmith, LangGraph allows for granular observability, meaning you can trace the latency
and token usage of a specific subgraph node 


How do you handle persistence (check-pointing) in subgraphs?

Answer: You typically provide a check-pointer to the parent graph.
LangGraph then automatically propagates that check-pointing capability to the child subgraphs 


In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
class SubState(TypedDict):
    input_text: str
    translated_text: str

In [ ]:
subgraph_llm = ChatOpenAI(model='gpt-4o')

In [ ]:
def translate_text(state: SubState):

    prompt = f"""Translate the following text to Hindi.Keep it natural and clear. Do not add extra content.Text:{state["input_text"]}""".strip()
    
    translated_text = subgraph_llm.invoke(prompt).content

    return {'translated_text': translated_text}



In [ ]:
subgraph_builder = StateGraph(SubState)

subgraph_builder.add_node('translate_text', translate_text)

subgraph_builder.add_edge(START, 'translate_text')
subgraph_builder.add_edge('translate_text', END)

subgraph = subgraph_builder.compile()

In [ ]:
class ParentState(TypedDict):
    question: str
    answer_eng: str
    answer_hin: str
    

In [ ]:
parent_llm = ChatOpenAI(model='gpt-4o-mini')

In [ ]:
def generate_answer(state: ParentState):

    answer = parent_llm.invoke(f"You are a helpful assistant. Answer clearly.\n\nQuestion: {state['question']}").content
    return {'answer_eng': answer}

In [ ]:
def translate_answer(state: ParentState):

    # call the subgraph
    result = subgraph.invoke({'input_text': state['answer_eng']})

    return {'answer_hin': result['translated_text']}

In [ ]:
parent_builder = StateGraph(ParentState)

parent_builder.add_node("answer", generate_answer)
parent_builder.add_node("translate", translate_answer)

parent_builder.add_edge(START, 'answer')
parent_builder.add_edge('answer', 'translate')
parent_builder.add_edge('translate', END)

In [ ]:
graph = parent_builder.compile()

graph

In [ ]:
graph.invoke({'question': 'What is quantum physics'})